### Libraries

In [86]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

In [88]:
print("Current working directory:", Path.cwd().parent.parent)
#check ya shabaaaab 

Current working directory: c:\Users\Mostafa\Desktop\Fall26\Fall26-MACHINE-LEARNING\project\hotel-pms-predictive-suite


### import data

In [89]:
path = Path.cwd().parent.parent
raw_datapath = path / "data" / "raw" / "H2.csv"
raw_csv = pd.read_csv(raw_datapath)
# Prevent Pandas from hiding columns with '...' in the terminal
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

## General Overview of the data

In [90]:
print("____Structure of the raw data dimensions and info:_____")
print("rows/samples count:", raw_csv.shape[0])
print("columns/features count:", raw_csv.shape[1])
print("++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++")
print("____Data Samples:_____")
print(raw_csv.sample(n=5))
print("++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++")

____Structure of the raw data dimensions and info:_____
rows/samples count: 79330
columns/features count: 31
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
____Data Samples:_____
       IsCanceled  LeadTime  ArrivalDateYear ArrivalDateMonth  ArrivalDateWeekNumber  ArrivalDateDayOfMonth  StaysInWeekendNights  StaysInWeekNights  Adults  Children  Babies       Meal Country MarketSegment DistributionChannel  IsRepeatedGuest  PreviousCancellations  PreviousBookingsNotCanceled  ReservedRoomType  AssignedRoomType  BookingChanges      DepositType        Agent      Company  DaysInWaitingList CustomerType     ADR  RequiredCarParkingSpaces  TotalOfSpecialRequests ReservationStatus ReservationStatusDate
56834           0         8             2016         December                     53                     30                     1                  2       2       1.0       0  BB            PRT     Corporate              Direct                1                      0                

## Statistical Summ up

In [91]:
print("summary of the raw data:")
print("++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++")
print(raw_csv.describe(include='all'))

summary of the raw data:
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
          IsCanceled      LeadTime  ArrivalDateYear ArrivalDateMonth  ArrivalDateWeekNumber  ArrivalDateDayOfMonth  StaysInWeekendNights  StaysInWeekNights        Adults      Children        Babies       Meal Country MarketSegment DistributionChannel  IsRepeatedGuest  PreviousCancellations  PreviousBookingsNotCanceled  ReservedRoomType  AssignedRoomType  BookingChanges      DepositType        Agent      Company  DaysInWaitingList CustomerType           ADR  RequiredCarParkingSpaces  TotalOfSpecialRequests ReservationStatus ReservationStatusDate
count   79330.000000  79330.000000     79330.000000            79330           79330.000000           79330.000000          79330.000000       79330.000000  79330.000000  79326.000000  79330.000000      79330   79306         79330               79330     79330.000000           79330.000000                 79330.000000             79330             79330    79

## Missing vals and duplicates:

In [ ]:
missing = raw_csv.isna().sum()
missing = missing[missing > 0]
missing_report = pd.DataFrame({"count": missing, "pct": (missing / len(raw_csv) * 100).round(2)})
print(missing_report.sort_values(by="count", ascending=False))
# check duplicates
duplicates = raw_csv.duplicated().sum()
print("++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++")


# 1. Strip whitespace from all string columns
str_cols = raw_csv.select_dtypes(include=['object', 'string']).columns
raw_csv[str_cols] = raw_csv[str_cols].apply(lambda s: s.str.strip())
# 2. standardize null values to np.nan
raw_csv = raw_csv.replace(['NULL', 'Undefined'], np.nan)

print(f"Number of duplicate rows : {duplicates}")

print("++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++")

                     count    pct
Company              75497  95.37
Agent                 8080  10.21
Country                 16   0.02
Children                 4   0.01
DistributionChannel      4   0.01
MarketSegment            2   0.00
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
Number of duplicate rows before cleaning: 25889
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++


### unlogical data  and stripping down useless data

In [104]:
# rows with Adults, Children, and Babies all equal to 0
print("count of rows before removing rows with Adults, Children, and Babies all equal to 0:", raw_csv.shape[0])
raw_csv = raw_csv[~((raw_csv['Adults'] == 0) & (raw_csv['Children'] == 0) & (raw_csv['Babies'] == 0))]
print("count of rows after removing rows with Adults, Children, and Babies all equal to 0:", raw_csv.shape[0],)
#drop duplicates
raw_csv = raw_csv.drop_duplicates()
print("Removed duplicates")


count of rows before removing rows with Adults, Children, and Babies all equal to 0: 53274
count of rows after removing rows with Adults, Children, and Babies all equal to 0: 53274
Removed duplicates


### usefull summary after cleaning

In [102]:
print("sample count after cleaning:", raw_csv.shape[0])

print("\n1. NUMERICAL SUMMARY (Transposed so no columns are hidden):")
print(raw_csv.describe().T[['count', 'mean', 'std', 'min', '50%', 'max']])

print("\n2. CATEGORICAL SUMMARY & UNIQUE VALUES:")
cat_cols = raw_csv.select_dtypes(include=['object']).columns
for col in cat_cols:
    unique_vals = raw_csv[col].unique()
    print(f"- {col} ({len(unique_vals)} unique): {unique_vals[:8]}") # Shows first 8 unique values

sample count after cleaning: 53274

1. NUMERICAL SUMMARY (Transposed so no columns are hidden):
                               count         mean        std     min     50%     max
IsCanceled                   53274.0     0.300991   0.458693     0.0     0.0     1.0
LeadTime                     53274.0    77.793257  82.112668     0.0    51.0   629.0
ArrivalDateYear              53274.0  2016.254120   0.659203  2015.0  2016.0  2017.0
ArrivalDateWeekNumber        53274.0    26.632973  13.579054     1.0    27.0    53.0
ArrivalDateDayOfMonth        53274.0    15.756560   8.802187     1.0    16.0    31.0
StaysInWeekendNights         53274.0     0.870725   0.904842     0.0     1.0    14.0
StaysInWeekNights            53274.0     2.271220   1.588220     0.0     2.0    34.0
Adults                       53274.0     1.881762   0.535634     0.0     2.0     4.0
Children                     53270.0     0.132232   0.442758     0.0     0.0     3.0
Babies                       53274.0     0.007358   0.

C:\Users\Mostafa\AppData\Local\Temp\ipykernel_29844\4059599251.py:7: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = raw_csv.select_dtypes(include=['object']).columns


In [105]:
raw_csv.to_csv(path/ "data"/ "cleaned"/ "H2_cleaned.csv", index=False)